# D7 Stage 2b Acceptance Notebook

This notebook is the independent sign-off evidence for the D7 Stage 2b five-call live critic probe. It deliberately does not repeat the fire-script summary. Instead, it independently rebuilds the aggregate evidence from source artifacts, audits the ledger, reparses raw model responses, rescans prompts/responses, and reconciles observed D7b behavior against the pre-registered expectations.

Primary gates covered:

1. Selection artifact gate
2. Commit-ordering gate
3. Per-call raw response gate
4. Prompt leakage gate
5. Ledger gate
6. Aggregate rebuild gate
7. Reconciliation gate
8. Stage 2c readiness gate


## 0. Setup

Locate the repository root, load the locked artifacts, and define small assertion helpers. All later sections derive their evidence from these artifacts rather than from Claude Code's console summary.

In [1]:
from __future__ import annotations

import hashlib
import json
import os
import re
import sqlite3
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from IPython.display import display

START_CWD = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (START_CWD, *START_CWD.parents) if (p / 'pyproject.toml').exists() and (p / 'agents').exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f'Could not locate repo root from {START_CWD}')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

BATCH_UUID = '5cf76668-47d1-48d7-bd90-db06d31982ed'
SELECTION_PATH = Path('docs/d7_stage2b/replay_candidates.json')
EXPECTATIONS_PATH = Path('docs/d7_stage2b/stage2b_expectations.md')
AGGREGATE_PATH = Path('docs/d7_stage2b/stage2b_batch_record.json')
PER_CALL_GLOB = 'docs/d7_stage2b/call_*_live_call_record.json'
LEDGER_PATH = Path('agents/spend_ledger.db')
PROMPT_TEMPLATE_PATH = Path('agents/critic/d7b_prompt.py')

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding='utf-8'))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

def git_commit_ts(path: Path) -> int:
    out = subprocess.check_output(['git', 'log', '-1', '--format=%ct', '--', str(path)], text=True).strip()
    if not out:
        raise AssertionError(f'No git commit timestamp for {path}')
    return int(out)

def git_commit_iso(path: Path) -> str:
    ts = git_commit_ts(path)
    return datetime.fromtimestamp(ts, tz=timezone.utc).isoformat().replace('+00:00', 'Z')

def parse_iso(value: str) -> datetime:
    if value.endswith('Z'):
        value = value[:-1] + '+00:00'
    return datetime.fromisoformat(value)

def approx_equal(a: float, b: float, tol: float = 1e-9) -> bool:
    return abs(float(a) - float(b)) <= tol

for p in (SELECTION_PATH, EXPECTATIONS_PATH, AGGREGATE_PATH, LEDGER_PATH, PROMPT_TEMPLATE_PATH):
    assert p.exists(), f'Missing required artifact: {p}'

selection = load_json(SELECTION_PATH)
expectations_text = EXPECTATIONS_PATH.read_text(encoding='utf-8')
aggregate = load_json(AGGREGATE_PATH)
per_call_paths = sorted(Path('docs/d7_stage2b').glob('call_*_live_call_record.json'))
per_calls = [load_json(p) for p in per_call_paths]

print('Repo root:', REPO_ROOT)
print('Per-call records:', [p.name for p in per_call_paths])
assert len(per_call_paths) == 5, 'Expected exactly five per-call records'


Repo root: /Users/yutianyang/Documents/GitHub/btc-alpha-pipeline
Per-call records: ['call_1_live_call_record.json', 'call_2_live_call_record.json', 'call_3_live_call_record.json', 'call_4_live_call_record.json', 'call_5_live_call_record.json']


## A. Scope Recap + Locked Inputs

Lock the artifact set and independently verify the anti-hindsight properties: stage identity, schema version, five selected candidates, expectations headers matching the selection JSON exactly, and commit ordering.

In [2]:
assert aggregate['stage_label'] == 'd7_stage2b'
assert aggregate['record_version'] == '1.0'
assert aggregate['batch_uuid'] == BATCH_UUID
assert selection['stage_label'] == 'd7_stage2b'
assert selection['record_version'] == '1.0'
assert selection['batch_uuid'] == BATCH_UUID
assert len(selection['candidates']) == 5

required_expectation_headers = [
    '## Anti-Hindsight Anchor',
    '## Aggregate Expectations Across All 5 Calls',
    '## Per-Candidate Expectations',
]
for header in required_expectation_headers:
    assert header in expectations_text, f'Missing expectations header: {header}'

candidate_headers = []
for i, c in enumerate(selection['candidates'], 1):
    header = f"### Candidate {i} — Position {c['position']}, {c['theme']}, {c['d7a_b_relationship_label']}"
    candidate_headers.append(header)
    assert header in expectations_text, f'Missing exact candidate header: {header}'

selection_ts = git_commit_ts(SELECTION_PATH)
expectations_ts = git_commit_ts(EXPECTATIONS_PATH)
fire_start_ts = int(parse_iso(aggregate['fire_timestamp_utc_start']).timestamp())
assert selection_ts < expectations_ts < fire_start_ts
assert aggregate['selection_commit_timestamp_utc'] == git_commit_iso(SELECTION_PATH)
assert aggregate['expectations_commit_timestamp_utc'] == git_commit_iso(EXPECTATIONS_PATH)

locked_inputs_df = pd.DataFrame([
    {'artifact': str(SELECTION_PATH), 'exists': SELECTION_PATH.exists(), 'sha256': sha256_file(SELECTION_PATH)},
    {'artifact': str(EXPECTATIONS_PATH), 'exists': EXPECTATIONS_PATH.exists(), 'sha256': sha256_file(EXPECTATIONS_PATH)},
    {'artifact': str(AGGREGATE_PATH), 'exists': AGGREGATE_PATH.exists(), 'sha256': sha256_file(AGGREGATE_PATH)},
    *[
        {'artifact': str(p), 'exists': p.exists(), 'sha256': sha256_file(p)}
        for p in per_call_paths
    ],
])
display(locked_inputs_df)
print('Commit ordering verified:', git_commit_iso(SELECTION_PATH), '<', git_commit_iso(EXPECTATIONS_PATH), '<', aggregate['fire_timestamp_utc_start'])


,artifact,exists,sha256
0,docs/d7_stage2b/replay_candidates.json,True,88ef1dc3a3928b9f44c103c81ed72da6a6a5ea266374e8...
1,docs/d7_stage2b/stage2b_expectations.md,True,5a2c51a0f0f31b00adc1300cbee86f9e355872175eb857...
2,docs/d7_stage2b/stage2b_batch_record.json,True,64dffba8e6ab6d97b9cba1b3d528511d1edaedb23435cb...
3,docs/d7_stage2b/call_1_live_call_record.json,True,011489dec5d6681d47f2dfd55b87673142bbc736f4989a...
4,docs/d7_stage2b/call_2_live_call_record.json,True,b7f04db59f5c1037dbdec0982ff2ac939a5d08a1514626...
5,docs/d7_stage2b/call_3_live_call_record.json,True,065fd18eab0fb13b1992d95532826417077b0e6e120d75...
6,docs/d7_stage2b/call_4_live_call_record.json,True,775ea1f1dca1138972d076df6acd68876abf4f0e4757ee...
7,docs/d7_stage2b/call_5_live_call_record.json,True,8c599d9624ff4627c204da6e485b99364444ebada0b500...


Commit ordering verified: 2026-04-18T12:24:47Z < 2026-04-18T13:11:31Z < 2026-04-18T13:12:21.588302Z


## B. Selection Artifact Audit

Rebuild the selection checks directly from `replay_candidates.json`. This proves the selection oracle did not silently mislabel candidates or violate diversity/order constraints.

In [3]:
candidates = selection['candidates']
positions = [c['position'] for c in candidates]
firing_orders = [c['firing_order'] for c in candidates]
themes = [c['theme'] for c in candidates]
buckets = [c['position_bucket'] for c in candidates]
hashes = [c['hypothesis_hash'] for c in candidates]
labels = [c['d7a_b_relationship_label'] for c in candidates]

assert positions == sorted(positions), 'Candidate positions are not ascending'
assert firing_orders == list(range(1, 6)), 'firing_order must be 1..5'
assert len(set(themes)) >= 3, 'Need at least three distinct themes'
assert {'early', 'mid', 'late'}.issubset(set(buckets)), 'Need early/mid/late coverage'
assert len(set(hashes)) == len(hashes), 'Duplicate hypothesis_hash detected'
assert 'agreement_expected' in labels
assert 'divergence_expected' in labels

def expected_label(prior_occurrences: int, max_overlap: int) -> str:
    if prior_occurrences > 0:
        return 'agreement_expected'
    if prior_occurrences == 0 and max_overlap <= 2:
        return 'divergence_expected'
    return 'neutral'

selection_rows = []
for c in candidates:
    recomputed = expected_label(c['factor_set_prior_occurrences'], c['max_overlap_with_priors'])
    selection_rows.append({
        'firing_order': c['firing_order'],
        'position': c['position'],
        'theme': c['theme'],
        'bucket': c['position_bucket'],
        'hash': c['hypothesis_hash'],
        'prior_occurrences': c['factor_set_prior_occurrences'],
        'max_overlap': c['max_overlap_with_priors'],
        'json_label': c['d7a_b_relationship_label'],
        'recomputed_label': recomputed,
        'label_match': recomputed == c['d7a_b_relationship_label'],
    })
selection_df = pd.DataFrame(selection_rows)
assert selection_df['label_match'].all(), 'Selection label mismatch detected'
display(selection_df)


,firing_order,position,theme,bucket,hash,prior_occurrences,max_overlap,json_label,recomputed_label,label_match
0,1,17,mean_reversion,early,17396a8e291dae75,0,0,divergence_expected,divergence_expected,True
1,2,73,volatility_regime,mid,ab3c1725e1cf4890,0,2,divergence_expected,divergence_expected,True
2,3,74,volume_divergence,mid,eb019d8da279abb2,0,2,divergence_expected,divergence_expected,True
3,4,97,mean_reversion,mid,1f8cbe2ba01c13a9,1,6,agreement_expected,agreement_expected,True
4,5,138,volatility_regime,late,9da35eee117da577,0,4,neutral,neutral,True


## C. Batch Record Integrity Audit

Rebuild the aggregate ordered sequences from the five per-call records. This is the central Stage 2b aggregate-record integrity check.

In [4]:
assert aggregate['completed_call_count'] == 5
assert aggregate['sequence_aborted'] is False
assert aggregate['abort_reason'] is None
assert aggregate['critic_statuses_in_call_order'] == ['ok'] * 5
assert aggregate['d7b_error_categories_in_call_order'] == [None] * 5
assert 'write_completed_at' in aggregate
assert parse_iso(aggregate['write_completed_at']) >= parse_iso(aggregate['fire_timestamp_utc_end'])

recomputed_reasoning_lengths = [len(c['critic_result'].get('d7b_reasoning') or '') for c in per_calls]
recomputed_actual_costs = [round(float(c['actual_cost_usd']), 6) for c in per_calls]
recomputed_input_tokens = [c['critic_result']['d7b_input_tokens'] for c in per_calls]
recomputed_output_tokens = [c['critic_result']['d7b_output_tokens'] for c in per_calls]

assert recomputed_reasoning_lengths == aggregate['reasoning_lengths_in_call_order']
assert recomputed_actual_costs == aggregate['actual_costs_in_call_order']
assert recomputed_input_tokens == aggregate['input_tokens_in_call_order']
assert recomputed_output_tokens == aggregate['output_tokens_in_call_order']
assert approx_equal(sum(recomputed_actual_costs), aggregate['total_actual_cost_usd'], tol=1e-6)
assert sum(recomputed_input_tokens) == aggregate['total_input_tokens']
assert sum(recomputed_output_tokens) == aggregate['total_output_tokens']

integrity_df = pd.DataFrame({
    'call': list(range(1, 6)),
    'reasoning_len_rebuilt': recomputed_reasoning_lengths,
    'reasoning_len_aggregate': aggregate['reasoning_lengths_in_call_order'],
    'actual_cost_rebuilt': recomputed_actual_costs,
    'actual_cost_aggregate': aggregate['actual_costs_in_call_order'],
    'input_tokens_rebuilt': recomputed_input_tokens,
    'input_tokens_aggregate': aggregate['input_tokens_in_call_order'],
    'output_tokens_rebuilt': recomputed_output_tokens,
    'output_tokens_aggregate': aggregate['output_tokens_in_call_order'],
})
display(integrity_df)


,call,reasoning_len_rebuilt,reasoning_len_aggregate,actual_cost_rebuilt,actual_cost_aggregate,input_tokens_rebuilt,input_tokens_aggregate,output_tokens_rebuilt,output_tokens_aggregate
0,1,1237,1237,0.011937,0.011937,1909,1909,414,414
1,2,1116,1116,0.014526,0.014526,3192,3192,330,330
2,3,1050,1050,0.015027,0.015027,3204,3204,361,361
3,4,1106,1106,0.016470,0.016470,3780,3780,342,342
4,5,1164,1164,0.018765,0.018765,4475,4475,356,356


## D. Per-Call Forensic Audit

Audit all five calls. For every raw response, independently parse JSON, verify exact schema, score ranges, reasoning length, forbidden-language scan, and refusal scan. For every prompt, independently re-run the leakage audit. Finally, verify each record links back to the selected candidate and physical raw payload files.

In [5]:
RAW_RESPONSE_KEYS = {
    'semantic_plausibility',
    'semantic_theme_alignment',
    'structural_variant_risk',
    'reasoning',
}

FORBIDDEN_TERMS = frozenset({
    'accept', 'reject', 'approve', 'approved', 'veto',
    'pass', 'fail', 'passing', 'failing',
    'recommend', 'recommended', 'recommendation',
    'should be used', 'should not be used',
    'do not use', 'keep', 'discard',
    'green light', 'red flag', 'go-ahead', 'go ahead',
})

REFUSAL_PATTERNS = (
    r'\bi cannot\b',
    r"\bi can't\b",
    r'\bunable to\b',
    r'\brefuse to\b',
    r'\bdecline to\b',
    r'\bcannot evaluate\b',
    r'\bcan not evaluate\b',
)

PROTECTED_TERMS = (
    'batch_id', 'batch_position', 'call_index', 'holdout',
    'validation_start', 'validation_end', 'validation_sharpe', 'validation_return',
    'test_start', 'test_end', 'backtest_result', 'sharpe', 'total_return',
    'max_drawdown', 'equity_curve', 'pnl', 'profit', 'approved_examples',
    'theme_coherence', 'structural_novelty', 'default_momentum_fallback',
    'complexity_appropriateness', 'empty_factor_set', 'thin_theme_momentum_bleed',
    'factor_set_in_top3_repeated', 'theme_anchor_missing',
    'description_length_near_limit', 'n_conditions_heavy',
)

UUID_RE = re.compile(r'[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}', re.IGNORECASE)
YEAR_RE = re.compile(r'\b(2022|2023|2024|2025|2026)-\d{2}-\d{2}')
DIRECTIVE_WORDS_RE = re.compile(
    r'\b(recommend|recommended|recommendation|approve|approved|reject|rejected|accept|accepted|pass\b|fail\b|passing|failing)\b',
    re.IGNORECASE,
)

def word_boundary_pattern(term: str) -> re.Pattern:
    escaped = re.escape(term).replace(r'\ ', r'\s+')
    return re.compile(rf'\b{escaped}\b', re.IGNORECASE)

FORBIDDEN_PATTERNS = {term: word_boundary_pattern(term) for term in FORBIDDEN_TERMS}
REFUSAL_REGEXES = [re.compile(pattern, re.IGNORECASE) for pattern in REFUSAL_PATTERNS]

def scan_forbidden(text: str) -> list[str]:
    return [term for term, pattern in FORBIDDEN_PATTERNS.items() if pattern.search(text)]

def scan_refusal(text: str) -> list[str]:
    return [pattern.pattern for pattern in REFUSAL_REGEXES if pattern.search(text)]

FENCE_RE = re.compile(r'^\s*```(?:json|JSON)?\s*\n(.*?)\n```\s*$', re.DOTALL)

def strip_fences_and_prose(text: str) -> str:
    text = text.strip()
    m = FENCE_RE.match(text)
    if m:
        text = m.group(1)
    first = text.find('{')
    if first < 0:
        return text
    depth = 0
    for i in range(first, len(text)):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[first:i + 1]
    return text

def strip_forbidden_language_block(prompt_text: str) -> str:
    pattern = re.compile(r'### Forbidden-language contract \(ENUMERATION BLOCK\).*?### Output contract', re.DOTALL)
    return pattern.sub('### Output contract', prompt_text)

def leakage_hits(prompt_text: str) -> list[str]:
    hits = []
    if UUID_RE.search(prompt_text):
        hits.append('uuid')
    if YEAR_RE.search(prompt_text):
        hits.append('year_prefixed_date')
    lower = prompt_text.lower()
    hits.extend([f'protected:{term}' for term in PROTECTED_TERMS if term.lower() in lower])
    stripped = strip_forbidden_language_block(prompt_text)
    m = DIRECTIVE_WORDS_RE.search(stripped)
    if m:
        hits.append(f'directive_outside_enumeration:{m.group(0)}')
    return hits

selection_by_order = {c['firing_order']: c for c in candidates}
forensic_rows = []

for rec in per_calls:
    order = rec['firing_order']
    cand = selection_by_order[order]
    raw_paths = rec['raw_payload_paths']
    response_path = Path(raw_paths['response'])
    prompt_path = Path(raw_paths['prompt'])
    critic_result_path = Path(raw_paths['critic_result'])
    assert response_path.exists(), f'Missing raw response: {response_path}'
    assert prompt_path.exists(), f'Missing prompt: {prompt_path}'
    assert critic_result_path.exists(), f'Missing critic result: {critic_result_path}'

    raw_response_text = response_path.read_text(encoding='utf-8')
    raw_payload = json.loads(strip_fences_and_prose(raw_response_text))
    assert set(raw_payload.keys()) == RAW_RESPONSE_KEYS
    reasoning = raw_payload['reasoning']
    scores = {k: raw_payload[k] for k in RAW_RESPONSE_KEYS if k != 'reasoning'}
    assert all(isinstance(v, (int, float)) and 0.0 <= float(v) <= 1.0 for v in scores.values())
    assert 100 <= len(reasoning) <= 1500
    forbidden_hits = scan_forbidden(reasoning)
    refusal_hits = scan_refusal(reasoning)
    assert forbidden_hits == []
    assert refusal_hits == []

    prompt_text = prompt_path.read_text(encoding='utf-8')
    prompt_leakage_hits = leakage_hits(prompt_text)
    assert prompt_leakage_hits == []

    assert rec['candidate_position'] == cand['position']
    assert rec['candidate_theme'] == cand['theme']
    assert rec['candidate_hypothesis_hash'] == cand['hypothesis_hash']
    assert rec['pre_registered_label'] == cand['d7a_b_relationship_label']
    assert rec['critic_status'] == 'ok'
    assert rec['leakage_audit_result']['status'] == 'pass'
    assert rec['forbidden_language_scan_result']['status'] == 'pass'
    assert rec['refusal_scan_result']['status'] == 'pass'

    forensic_rows.append({
        'call': order,
        'position': cand['position'],
        'theme': cand['theme'],
        'raw_response_schema_ok': True,
        'reasoning_length': len(reasoning),
        'score_min': min(scores.values()),
        'score_max': max(scores.values()),
        'forbidden_hits': forbidden_hits,
        'refusal_hits': refusal_hits,
        'prompt_leakage_hits': prompt_leakage_hits,
        'record_linkage_ok': True,
    })

forensic_df = pd.DataFrame(forensic_rows)
display(forensic_df)


,call,position,theme,raw_response_schema_ok,reasoning_length,score_min,score_max,forbidden_hits,refusal_hits,prompt_leakage_hits,record_linkage_ok
0,1,17,mean_reversion,True,1237,0.75,0.85,[],[],[],True
1,2,73,volatility_regime,True,1116,0.75,0.85,[],[],[],True
2,3,74,volume_divergence,True,1050,0.65,0.85,[],[],[],True
3,4,97,mean_reversion,True,1106,0.75,0.95,[],[],[],True
4,5,138,volatility_regime,True,1164,0.15,0.90,[],[],[],True


## E. Ledger Audit

Query SQLite directly. The ledger must contain exactly five Stage 2b live critic rows, with correct tags, completed status, positive actual cost, token counts, strict sequential timing, and exact agreement with per-call records and the aggregate total.

In [6]:
conn = sqlite3.connect(LEDGER_PATH)
ledger_rows = conn.execute(
    """
    SELECT id, batch_id, api_call_kind, backend_kind, call_role, status,
           estimated_cost, actual_cost, input_tokens, output_tokens,
           created_at_utc, completed_at_utc, notes
    FROM ledger
    WHERE batch_id = ?
      AND backend_kind = 'd7b_critic'
      AND notes LIKE 'Stage 2b live%'
    ORDER BY created_at_utc
    """,
    (BATCH_UUID,),
).fetchall()
conn.close()

ledger_cols = [
    'id', 'batch_id', 'api_call_kind', 'backend_kind', 'call_role', 'status',
    'estimated_cost', 'actual_cost', 'input_tokens', 'output_tokens',
    'created_at_utc', 'completed_at_utc', 'notes',
]
ledger_df = pd.DataFrame(ledger_rows, columns=ledger_cols)
assert len(ledger_df) == 5
assert (ledger_df['api_call_kind'] == 'd7b_critic_live').all()
assert (ledger_df['call_role'] == 'critique').all()
assert (ledger_df['status'] == 'completed').all()
assert ledger_df['estimated_cost'].map(lambda x: approx_equal(x, 0.05)).all()
assert (ledger_df['actual_cost'] > 0).all()
assert ledger_df['input_tokens'].notna().all()
assert ledger_df['output_tokens'].notna().all()

created = ledger_df['created_at_utc'].map(parse_iso).tolist()
completed = ledger_df['completed_at_utc'].map(parse_iso).tolist()
assert all(created[i] < created[i + 1] for i in range(4))
assert all(completed[i] < completed[i + 1] for i in range(4))
assert all(completed[i] < created[i + 1] for i in range(4)), 'Ledger rows overlap; expected sequential calls'

per_call_by_order = {rec['firing_order']: rec for rec in per_calls}
ledger_cost_sum = round(float(ledger_df['actual_cost'].sum()), 6)
assert approx_equal(ledger_cost_sum, aggregate['total_actual_cost_usd'], tol=1e-6)

for _, row in ledger_df.iterrows():
    m = re.search(r'firing_order=(\d+)', row['notes'])
    assert m, f'Missing firing_order in notes: {row["notes"]}'
    order = int(m.group(1))
    rec = per_call_by_order[order]
    assert rec['ledger_row']['row_id'] == row['id']
    assert approx_equal(rec['actual_cost_usd'], row['actual_cost'], tol=1e-9)
    assert rec['critic_result']['d7b_input_tokens'] == row['input_tokens']
    assert rec['critic_result']['d7b_output_tokens'] == row['output_tokens']

display(ledger_df)


,id,batch_id,api_call_kind,backend_kind,call_role,status,estimated_cost,actual_cost,input_tokens,output_tokens,created_at_utc,completed_at_utc,notes
0,8cb56cd8-ab75-43fb-b267-c3f4dd5f8bd9,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.011937,1909,414,2026-04-18T13:12:21.589Z,2026-04-18T13:12:32.216Z,"Stage 2b live, position=17, firing_order=1"
1,145a01c7-c8e0-426a-bb58-3e5bacfa6ce6,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.014526,3192,330,2026-04-18T13:12:37.223Z,2026-04-18T13:12:46.223Z,"Stage 2b live, position=73, firing_order=2"
2,d1e1e0a8-fb91-4e24-89af-ea5aff14e997,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.015027,3204,361,2026-04-18T13:12:51.233Z,2026-04-18T13:13:02.697Z,"Stage 2b live, position=74, firing_order=3"
3,8200dadd-dca5-4fc9-a2ef-6d888ee302f6,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.016470,3780,342,2026-04-18T13:13:07.708Z,2026-04-18T13:13:18.779Z,"Stage 2b live, position=97, firing_order=4"
4,6b188b09-39a7-49c7-9cba-a5149878c9ca,5cf76668-47d1-48d7-bd90-db06d31982ed,d7b_critic_live,d7b_critic,critique,completed,0.05,0.018765,4475,356,2026-04-18T13:13:23.803Z,2026-04-18T13:13:34.341Z,"Stage 2b live, position=138, firing_order=5"


## F. Cost Model Audit

Independently tabulate context size, token usage, and cost accumulation. Stage 2b should stay safely under both the per-call and total cost ceilings, and actual spend should remain below the conservative estimate.

In [7]:
cost_rows = []
for rec in per_calls:
    cost_rows.append({
        'call': rec['firing_order'],
        'position': rec['candidate_position'],
        'prior_factor_sets_count': rec['prior_factor_sets_count'],
        'prompt_chars': rec['prompt_chars'],
        'input_tokens': rec['critic_result']['d7b_input_tokens'],
        'output_tokens': rec['critic_result']['d7b_output_tokens'],
        'estimated_cost': rec['cost']['estimated_usd'],
        'actual_cost': rec['actual_cost_usd'],
    })
cost_df = pd.DataFrame(cost_rows)
assert (cost_df['actual_cost'] < 0.05).all()
assert aggregate['total_actual_cost_usd'] < 0.10
assert aggregate['total_actual_cost_usd'] <= aggregate['total_estimated_cost_usd']
assert cost_df['actual_cost'].is_monotonic_increasing
assert cost_df['input_tokens'].is_monotonic_increasing
actual_to_estimated_ratio = aggregate['total_actual_cost_usd'] / aggregate['total_estimated_cost_usd']
display(cost_df)
print('total_actual_cost_usd:', aggregate['total_actual_cost_usd'])
print('total_estimated_cost_usd:', aggregate['total_estimated_cost_usd'])
print('actual_to_estimated_ratio:', round(actual_to_estimated_ratio, 4))


,call,position,prior_factor_sets_count,prompt_chars,input_tokens,output_tokens,estimated_cost,actual_cost
0,1,17,16,5810,1909,414,0.05,0.011937
1,2,73,60,8429,3192,330,0.05,0.014526
2,3,74,61,8422,3204,361,0.05,0.015027
3,4,97,76,9663,3780,342,0.05,0.016470
4,5,138,99,11077,4475,356,0.05,0.018765


total_actual_cost_usd: 0.076725
total_estimated_cost_usd: 0.25
actual_to_estimated_ratio: 0.3069


## G. Agreement/Divergence Reconciliation Audit

Recompute the scientific core independently: map each pre-registered label to its expected structural-variant-risk direction, compare with observed D7b scores, and verify the aggregate reconciliation field was calculated correctly.

In [8]:
def expected_direction(label: str) -> str:
    if label == 'divergence_expected':
        return 'low'
    if label == 'agreement_expected':
        return 'high'
    return 'none'

def observed_direction(svr: float) -> str:
    return 'high' if svr >= 0.5 else 'low'

def reconcile(label: str, svr: float) -> str:
    expected = expected_direction(label)
    observed = observed_direction(svr)
    if expected == 'none':
        return 'n/a'
    return 'supports' if expected == observed else 'contradict'

recon_rows = []
for rec in per_calls:
    order = rec['firing_order']
    label = rec['pre_registered_label']
    svr = rec['critic_result']['d7b_llm_scores']['structural_variant_risk']
    row = {
        'call': order,
        'label': label,
        'svr': svr,
        'expected': expected_direction(label),
        'observed': observed_direction(svr),
        'reconcile': reconcile(label, svr),
    }
    agg_recon = aggregate['agreement_divergence_reconciliation_by_call'][str(order)]
    expected_bool = None if row['reconcile'] == 'n/a' else row['reconcile'] == 'supports'
    assert agg_recon['pre_registered_label'] == label
    assert agg_recon['d7b_structural_variant_risk'] == svr
    assert agg_recon['observed_consistent_with_label'] is expected_bool
    recon_rows.append(row)

recon_df = pd.DataFrame(recon_rows)
assert list(recon_df['reconcile']) == ['contradict', 'contradict', 'contradict', 'supports', 'n/a']
display(recon_df)


,call,label,svr,expected,observed,reconcile
0,1,divergence_expected,0.85,low,high,contradict
1,2,divergence_expected,0.85,low,high,contradict
2,3,divergence_expected,0.65,low,high,contradict
3,4,agreement_expected,0.95,high,high,supports
4,5,neutral,0.15,none,low,n/a


## H. Expectations Reconciliation

Bring the pre-registered expectations file back into the analysis. This section extracts each candidate section and compares its locked directional expectation with the observed behavior.

In [9]:
def section_for_header(text: str, header: str) -> str:
    start = text.index(header)
    next_match = re.search(r'\n### Candidate \d+ — ', text[start + len(header):])
    if next_match:
        end = start + len(header) + next_match.start()
        return text[start:end]
    return text[start:]

expectation_rows = []
for i, cand in enumerate(candidates, 1):
    header = candidate_headers[i - 1]
    section = section_for_header(expectations_text, header)
    label = cand['d7a_b_relationship_label']
    svr = per_call_by_order[i]['critic_result']['d7b_llm_scores']['structural_variant_risk']
    expected = expected_direction(label)
    observed = observed_direction(svr)
    if expected == 'none':
        verdict = 'match'
        evidence = 'neutral section made no directional prediction'
    elif expected == observed:
        verdict = 'match'
        evidence = f'expected {expected}; observed {observed}'
    else:
        verdict = 'mismatch'
        evidence = f'expected {expected}; observed {observed}'
    assert cand['hypothesis_hash'] in section
    if label == 'divergence_expected':
        assert 'structural_variant_risk < 0.5' in section
    if label == 'agreement_expected':
        assert 'structural_variant_risk >= 0.5' in section
    if label == 'neutral':
        assert 'no' in section.lower() and 'direction' in section.lower()
    expectation_rows.append({
        'call': i,
        'header': header,
        'hash': cand['hypothesis_hash'],
        'pre_registered_label': label,
        'svr': svr,
        'expected_direction': expected,
        'observed_direction': observed,
        'expectations_verdict': verdict,
        'evidence': evidence,
    })

expectation_df = pd.DataFrame(expectation_rows)
display(expectation_df)


,call,header,hash,pre_registered_label,svr,expected_direction,observed_direction,expectations_verdict,evidence
0,1,"### Candidate 1 — Position 17, mean_reversion,...",17396a8e291dae75,divergence_expected,0.85,low,high,mismatch,expected low; observed high
1,2,"### Candidate 2 — Position 73, volatility_regi...",ab3c1725e1cf4890,divergence_expected,0.85,low,high,mismatch,expected low; observed high
2,3,"### Candidate 3 — Position 74, volume_divergen...",eb019d8da279abb2,divergence_expected,0.65,low,high,mismatch,expected low; observed high
3,4,"### Candidate 4 — Position 97, mean_reversion,...",1f8cbe2ba01c13a9,agreement_expected,0.95,high,high,match,expected high; observed high
4,5,"### Candidate 5 — Position 138, volatility_reg...",9da35eee117da577,neutral,0.15,none,low,match,neutral section made no directional prediction


## I. Hard-Fail Gates Audit

Explicit pass/fail checklist for the Stage 2b hard-fail gates. Most gates are fully reconstructed from artifacts. Gates that are inherently pre-fire only are supported by the successful live execution plus persisted record evidence.

In [10]:
aggregate_key_order = list(json.loads(AGGREGATE_PATH.read_text(encoding='utf-8')).keys())
current_selection_sha = sha256_file(SELECTION_PATH)
current_expectations_sha = sha256_file(EXPECTATIONS_PATH)
current_prompt_template_sha = sha256_file(PROMPT_TEMPLATE_PATH)

hard_gates = [
    ('HG1', 'selection JSON exists/parses and contains exactly five ordered candidates', len(candidates) == 5 and firing_orders == [1, 2, 3, 4, 5]),
    ('HG2', 'selection JSON SHA-256 captured in aggregate and matches current file', aggregate['selection_json_sha256'] == current_selection_sha),
    ('HG2b', 'selection tier invariants satisfied', selection['selection_tier'] == 0 and selection['selection_warnings'] == []),
    ('HG3', 'expectations file exists and is committed', EXPECTATIONS_PATH.exists() and expectations_ts is not None),
    ('HG4', 'expectations required headers present', all(h in expectations_text for h in required_expectation_headers)),
    ('HG4b', 'candidate expectation headers match selection exactly', all(h in expectations_text for h in candidate_headers)),
    ('HG5', 'selection commit < expectations commit < fire time', selection_ts < expectations_ts < fire_start_ts),
    ('HG6', 'prompt template hash captured and matches current template', aggregate['d7b_prompt_template_sha256'] == current_prompt_template_sha),
    ('HG6b', 'no prompt-template-drift abort occurred mid-sequence', aggregate['abort_reason'] != 'prompt_template_mutated_mid_run'),
    ('HG7', 'expectations SHA-256 captured and matches current file', aggregate['expectations_file_sha256'] == current_expectations_sha),
    ('HG8', 'aggregate record written once with successful live-run evidence', AGGREGATE_PATH.exists() and aggregate['fire_script_command'].endswith('--confirm-live')),
    ('HG9', 'per-call candidate identity matches selection', forensic_df['record_linkage_ok'].all()),
    ('HG10', 'all prompts pass independent leakage audit', forensic_df['prompt_leakage_hits'].map(len).eq(0).all()),
    ('HG11', 'all raw responses parse with exact four-key schema', forensic_df['raw_response_schema_ok'].all()),
    ('HG12', 'all scores/rationales are in contract bounds', (forensic_df['score_min'] >= 0).all() and (forensic_df['score_max'] <= 1).all() and forensic_df['reasoning_length'].between(100, 1500).all()),
    ('HG13', 'ledger rows are tagged and finalized correctly', len(ledger_df) == 5 and (ledger_df['status'] == 'completed').all()),
    ('HG14', 'all five per-call records exist', len(per_call_paths) == 5 and all(p.exists() for p in per_call_paths)),
    ('HG15', 'all prompt/response/critic_result raw payloads exist', all(Path(v).exists() for rec in per_calls for v in [rec['raw_payload_paths']['prompt'], rec['raw_payload_paths']['response'], rec['raw_payload_paths']['critic_result']])),
    ('HG16', 'write_completed_at exists and is last aggregate key', aggregate_key_order[-1] == 'write_completed_at'),
    ('HG17', 'each per-call actual cost is below $0.05 ceiling', (cost_df['actual_cost'] < 0.05).all()),
    ('HG18', 'total actual cost is below $0.10 cap', aggregate['total_actual_cost_usd'] < 0.10),
    ('HG19', 'sequence completed without abort and all statuses ok', aggregate['sequence_aborted'] is False and aggregate['critic_statuses_in_call_order'] == ['ok'] * 5),
    ('HG20', 'selection JSON did not drift by sequence end', aggregate.get('hg20_drift_detected') is not True and aggregate['selection_json_sha256'] == current_selection_sha),
]

hard_gate_df = pd.DataFrame(hard_gates, columns=['gate', 'evidence', 'pass'])
assert hard_gate_df['pass'].all(), hard_gate_df.loc[~hard_gate_df['pass']]
display(hard_gate_df)


,gate,evidence,pass
0,HG1,selection JSON exists/parses and contains exac...,True
1,HG2,selection JSON SHA-256 captured in aggregate a...,True
2,HG2b,selection tier invariants satisfied,True
3,HG3,expectations file exists and is committed,True
4,HG4,expectations required headers present,True
5,HG4b,candidate expectation headers match selection ...,True
6,HG5,selection commit < expectations commit < fire ...,True
7,HG6,prompt template hash captured and matches curr...,True
8,HG6b,no prompt-template-drift abort occurred mid-se...,True
9,HG7,expectations SHA-256 captured and matches curr...,True


## J. Stage 2c Green-Light Assessment

The notebook does not sign off by itself, but it gives the structured evidence for the four Stage 2c readiness criteria.

In [11]:
green_lights = [
    ('actual <= estimated', aggregate['total_actual_cost_usd'] <= aggregate['total_estimated_cost_usd'], f"{aggregate['total_actual_cost_usd']} <= {aggregate['total_estimated_cost_usd']}"),
    ('all 5 reasoning strings under 1500 chars', max(aggregate['reasoning_lengths_in_call_order']) < 1500, str(aggregate['reasoning_lengths_in_call_order'])),
    ('divergence pattern or insufficient signal', (recon_df['reconcile'] == 'contradict').iloc[:3].all(), '3/3 divergence_expected candidates observed high structural_variant_risk'),
    ('no recurring error class', aggregate['d7b_error_categories_in_call_order'] == [None] * 5, str(aggregate['d7b_error_categories_in_call_order'])),
]
green_df = pd.DataFrame(green_lights, columns=['criterion', 'pass', 'evidence'])
assert green_df['pass'].all(), green_df.loc[~green_df['pass']]
display(green_df)

final_verdict = {
    'selection_artifact_gate': True,
    'commit_ordering_gate': selection_ts < expectations_ts < fire_start_ts,
    'per_call_response_gate': forensic_df['raw_response_schema_ok'].all() and forensic_df['forbidden_hits'].map(len).eq(0).all() and forensic_df['refusal_hits'].map(len).eq(0).all(),
    'prompt_leakage_gate': forensic_df['prompt_leakage_hits'].map(len).eq(0).all(),
    'ledger_gate': len(ledger_df) == 5 and approx_equal(ledger_cost_sum, aggregate['total_actual_cost_usd'], tol=1e-6),
    'aggregate_rebuild_gate': recomputed_reasoning_lengths == aggregate['reasoning_lengths_in_call_order'] and recomputed_actual_costs == aggregate['actual_costs_in_call_order'],
    'reconciliation_gate': list(recon_df['reconcile']) == ['contradict', 'contradict', 'contradict', 'supports', 'n/a'],
    'stage2c_readiness_gate': green_df['pass'].all(),
}
verdict_df = pd.DataFrame(final_verdict.items(), columns=['gate', 'pass'])
assert verdict_df['pass'].all(), verdict_df.loc[~verdict_df['pass']]
display(verdict_df)
print('D7 Stage 2b acceptance notebook gates: 8/8 PASS')


,criterion,pass,evidence
0,actual <= estimated,True,0.076725 <= 0.25
1,all 5 reasoning strings under 1500 chars,True,"[1237, 1116, 1050, 1106, 1164]"
2,divergence pattern or insufficient signal,True,3/3 divergence_expected candidates observed hi...
3,no recurring error class,True,"[None, None, None, None, None]"


,gate,pass
0,selection_artifact_gate,True
1,commit_ordering_gate,True
2,per_call_response_gate,True
3,prompt_leakage_gate,True
4,ledger_gate,True
5,aggregate_rebuild_gate,True
6,reconciliation_gate,True
7,stage2c_readiness_gate,True


D7 Stage 2b acceptance notebook gates: 8/8 PASS
